# Prediksi Machine Failure dengan XGBoost (AI4I 2020) + MLflow

Analogi singkat: model ini seperti teknisi yang dilatih membaca 4 sensor (suhu, putaran, torsi, keausan alat) lalu memutuskan "mesin ini bakal gagal atau aman?". Kita juga memberinya 3 "kalkulator bantu" (selisih suhu, daya, keausan x torsi) supaya ia tidak perlu menebak rumusnya sendiri.

Alur: load data (ClickHouse), feature engineering, split (train/val/test), training baseline + SMOTE, tuning threshold, evaluasi, **kirim ke MLflow (tracking + model registry)**, inference 1 data baru dari registry.

Semua konfigurasi dibaca dari environment variable (lihat sel konfigurasi), sehingga notebook yang sama bisa dijalankan manual maupun lewat `papermill` di pipeline CI/CD.

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.xgboost
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             average_precision_score, roc_auc_score,
                             precision_score, recall_score, f1_score,
                             precision_recall_curve, ConfusionMatrixDisplay)
from xgboost import XGBClassifier

RANDOM_STATE = 42

# ---------------- Konfigurasi (override lewat environment variable) ----------------
DATA_SOURCE = os.getenv("DATA_SOURCE", "clickhouse")        # "clickhouse" | "csv"
DATA_PATH   = os.getenv("DATA_PATH", "../data/ai4i2020.csv") # dipakai bila DATA_SOURCE=csv

CH_HOST     = os.getenv("CH_HOST", "localhost")
CH_PORT     = int(os.getenv("CH_PORT", "8123"))             # HTTP port
CH_USER     = os.getenv("CH_USER", "default")
CH_PASSWORD = os.getenv("CH_PASSWORD", "")
CH_DATABASE = os.getenv("CH_DATABASE", "default")
CH_TABLE    = os.getenv("CH_TABLE", "ai4i_2020")

MLFLOW_TRACKING_URI   = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
MLFLOW_EXPERIMENT     = os.getenv("MLFLOW_EXPERIMENT", "machine-failure-ai4i2020")
REGISTERED_MODEL_NAME = os.getenv("MLFLOW_MODEL_NAME", "machine_failure_xgb")
MODEL_ALIAS           = os.getenv("MLFLOW_MODEL_ALIAS", "champion")
MODEL_TO_REGISTER     = os.getenv("MODEL_TO_REGISTER", "auto")  # auto | baseline | smote
GIT_SHA               = os.getenv("GIT_SHA", "local")

## 1. Load data

Sumber utama: tabel ClickHouse hasil DAG Airflow (Oracle -> ClickHouse). Untuk eksperimen lokal, set `DATA_SOURCE=csv`.

In [ ]:
COLUMNS = ["udi", "product_id", "type", "air_temp_k", "process_temp_k",
           "rpm", "torque_nm", "tool_wear_min", "machine_failure",
           "twf", "hdf", "pwf", "osf", "rnf"]
INT_COLS   = ["udi", "rpm", "tool_wear_min", "machine_failure", "twf", "hdf", "pwf", "osf", "rnf"]
FLOAT_COLS = ["air_temp_k", "process_temp_k", "torque_nm"]


def load_from_clickhouse() -> pd.DataFrame:
    import clickhouse_connect
    client = clickhouse_connect.get_client(
        host=CH_HOST, port=CH_PORT, username=CH_USER,
        password=CH_PASSWORD, database=CH_DATABASE)
    return client.query_df(f"SELECT {', '.join(COLUMNS)} FROM {CH_TABLE} ORDER BY udi")


def load_from_csv() -> pd.DataFrame:
    # File CSV memakai BOM di awal, jadi pakai utf-8-sig agar nama kolom pertama tidak rusak
    d = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
    d.columns = COLUMNS
    return d


df = load_from_clickhouse() if DATA_SOURCE == "clickhouse" else load_from_csv()
df = df.astype({**{c: "int64" for c in INT_COLS}, **{c: "float64" for c in FLOAT_COLS}})

print("Sumber data:", DATA_SOURCE)
print(df.shape)
df.head()

In [ ]:
print(df.isna().sum().sum(), "nilai kosong")
print(df["machine_failure"].value_counts())
print(f"Rasio failure: {df['machine_failure'].mean():.2%}")

## 2. Feature engineering

Kolom `twf`, `hdf`, `pwf`, `osf`, `rnf` **dibuang** karena itu bocoran label (failure = 1 kalau salah satunya 1). `udi` dan `product_id` hanya identifier.

Fitur turunan meniru aturan fisik dari keterangan dataset:
- `temp_diff_k`: selisih suhu proses dan udara (aturan HDF)
- `power_w`: torsi x rpm x 2pi/60 (aturan PWF)
- `wear_torque`: keausan alat x torsi (aturan OSF)

In [ ]:
TYPE_MAP = {"L": 0, "M": 1, "H": 2}

def make_features(d: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=d.index)
    out["type_enc"]       = d["type"].map(TYPE_MAP)
    out["air_temp_k"]     = d["air_temp_k"]
    out["process_temp_k"] = d["process_temp_k"]
    out["rpm"]            = d["rpm"]
    out["torque_nm"]      = d["torque_nm"]
    out["tool_wear_min"]  = d["tool_wear_min"]
    out["temp_diff_k"]    = d["process_temp_k"] - d["air_temp_k"]
    out["power_w"]        = d["torque_nm"] * d["rpm"] * 2 * np.pi / 60
    out["wear_torque"]    = d["tool_wear_min"] * d["torque_nm"]
    return out

X = make_features(df)
y = df["machine_failure"]
FEATURES = list(X.columns)
X.head()

## 3. Split data

Stratified supaya proporsi failure sama di tiap bagian. Pembagian: train 64%, validation 16% (untuk early stopping dan tuning threshold), test 20% (hanya disentuh sekali di akhir).

In [ ]:
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.20, stratify=y_tmp, random_state=RANDOM_STATE)

for name, part in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s}: {len(part):5d} baris, failure {part.mean():.2%}")

### 3.1 Penanganan Imbalance dengan SMOTE

Karena data failure sangat tidak seimbang (hanya ~3.39%), kita akan menggunakan SMOTE (Synthetic Minority Over-sampling Technique) untuk membuat sampel failure sintetis baru pada **data training saja** agar model tidak bias.

In [ ]:
from imblearn.over_sampling import SMOTE

# Definisikan SMOTE
smote = SMOTE(random_state=RANDOM_STATE)

# Fit SMOTE hanya pada training set
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Sebelum SMOTE (Train):", np.bincount(y_train))
print("Setelah SMOTE (Train):", np.bincount(y_train_res))
print(f"Rasio failure setelah SMOTE: {y_train_res.mean():.2%}")

## 4. Training XGBoost

In [ ]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print("scale_pos_weight:", round(scale_pos_weight, 2))

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    early_stopping_rounds=30,
    tree_method="hist",
    random_state=RANDOM_STATE,
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print("Best iteration:", model.best_iteration)

### 4.1 Training XGBoost dengan Data Hasil SMOTE

Sekarang kita latih XGBoost menggunakan `X_train_res` dan `y_train_res`. Karena data sudah seimbang, kita tidak memerlukan parameter `scale_pos_weight` lagi.

In [ ]:
model_smote = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",
    early_stopping_rounds=30,
    tree_method="hist",
    random_state=RANDOM_STATE,
)

# Latih model dengan data latih hasil resampling SMOTE, validasi tetap menggunakan data val asli
model_smote.fit(X_train_res, y_train_res, eval_set=[(X_val, y_val)], verbose=False)
print("Best iteration (SMOTE):", model_smote.best_iteration)

### 5.1 Tuning Threshold & Evaluasi Model SMOTE

In [ ]:
val_proba_smote = model_smote.predict_proba(X_val)[:, 1]
prec_s, rec_s, thr_s = precision_recall_curve(y_val, val_proba_smote)
f1_s = 2 * prec_s * rec_s / (prec_s + rec_s + 1e-9)
best_s = np.argmax(f1_s[:-1])
THRESHOLD_SMOTE = float(thr_s[best_s])
print(f"Threshold terbaik (SMOTE): {THRESHOLD_SMOTE:.3f}  (F1 val = {f1_s[best_s]:.3f})")

test_proba_smote = model_smote.predict_proba(X_test)[:, 1]
test_pred_smote = (test_proba_smote >= THRESHOLD_SMOTE).astype(int)

print("\n--- EVALUASI MODEL SMOTE PADA TEST SET ---")
print(classification_report(y_test, test_pred_smote, target_names=["Normal", "Failure"], digits=3))
print("PR-AUC :", round(average_precision_score(y_test, test_proba_smote), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, test_proba_smote), 4))

## 5. Tuning threshold

Threshold 0,5 jarang ideal untuk data tidak seimbang. Kita cari threshold dengan F1 terbaik di **validation set**, bukan di test set.

In [ ]:
val_proba = model.predict_proba(X_val)[:, 1]
prec, rec, thr = precision_recall_curve(y_val, val_proba)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
best = np.argmax(f1[:-1])
THRESHOLD = float(thr[best])
print(f"Threshold terbaik: {THRESHOLD:.3f}  (F1 val = {f1[best]:.3f})")

## 6. Evaluasi di test set

In [ ]:
test_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= THRESHOLD).astype(int)

print(classification_report(y_test, test_pred, target_names=["Normal", "Failure"], digits=3))
print("PR-AUC :", round(average_precision_score(y_test, test_proba), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 4))

ConfusionMatrixDisplay(confusion_matrix(y_test, test_pred),
                       display_labels=["Normal", "Failure"]).plot(cmap="Blues")
plt.title("Confusion matrix (test)")
plt.show()

## 7. Feature importance

In [ ]:
imp = pd.Series(model.get_booster().get_score(importance_type="gain"))
imp = imp.reindex(FEATURES).fillna(0).sort_values()
imp.plot(kind="barh", figsize=(7, 4), title="Feature importance (gain)")
plt.tight_layout()
plt.show()

## 8. Kirim ke MLflow

Kedua model (baseline dan SMOTE) dicatat sebagai run terpisah: parameter, metrik validation + test, plot, dan model itu sendiri (dengan signature). Satu model dipilih sebagai **champion**, didaftarkan ke Model Registry, lalu diberi alias `champion`. FastAPI memuat model lewat alias ini, jadi tidak ada file model yang di-hardcode.

Pemilihan otomatis (`MODEL_TO_REGISTER=auto`) memakai **F1 di validation set**, bukan test set, supaya test set tetap bersih. Threshold dan daftar fitur disimpan sebagai tag pada model version agar API memakai konfigurasi yang sama persis dengan training.

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
client = MlflowClient()

LOG_PARAM_KEYS = ["n_estimators", "learning_rate", "max_depth", "subsample", "colsample_bytree",
                  "scale_pos_weight", "eval_metric", "early_stopping_rounds", "tree_method"]


def split_metrics(m, thr):
    """Precision/recall/F1 pada threshold terpilih + PR-AUC/ROC-AUC, untuk val dan test."""
    out = {}
    for split, (Xs, ys) in {"val": (X_val, y_val), "test": (X_test, y_test)}.items():
        proba = m.predict_proba(Xs)[:, 1]
        pred = (proba >= thr).astype(int)
        out[f"{split}_precision"] = precision_score(ys, pred, zero_division=0)
        out[f"{split}_recall"]    = recall_score(ys, pred, zero_division=0)
        out[f"{split}_f1"]        = f1_score(ys, pred, zero_division=0)
        out[f"{split}_pr_auc"]    = average_precision_score(ys, proba)
        out[f"{split}_roc_auc"]   = roc_auc_score(ys, proba)
    return {k: float(v) for k, v in out.items()}


def make_plots(m, thr):
    pred = (m.predict_proba(X_test)[:, 1] >= thr).astype(int)
    fig_cm, ax = plt.subplots(figsize=(4.5, 4))
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                           display_labels=["Normal", "Failure"]).plot(cmap="Blues", ax=ax)
    ax.set_title("Confusion matrix (test)")
    fig_cm.tight_layout()

    fig_imp, ax2 = plt.subplots(figsize=(7, 4))
    imp = pd.Series(m.get_booster().get_score(importance_type="gain"))
    imp.reindex(FEATURES).fillna(0).sort_values().plot(kind="barh", ax=ax2,
                                                       title="Feature importance (gain)")
    fig_imp.tight_layout()
    return fig_cm, fig_imp


def log_variant(variant, m, thr, train_rows, train_failure_ratio):
    meta = {"variant": variant, "features": FEATURES, "type_map": TYPE_MAP, "threshold": thr}
    with mlflow.start_run(run_name=f"xgb_{variant}") as run:
        mlflow.set_tags({"variant": variant, "data_source": DATA_SOURCE,
                         "git_sha": GIT_SHA, "framework": "xgboost"})
        params = {k: v for k, v in m.get_params().items() if k in LOG_PARAM_KEYS and v is not None}
        params.update(best_iteration=m.best_iteration, threshold=round(thr, 6),
                      train_rows=train_rows, train_failure_ratio=round(train_failure_ratio, 4),
                      n_features=len(FEATURES), random_state=RANDOM_STATE)
        mlflow.log_params(params)

        metrics = split_metrics(m, thr)
        mlflow.log_metrics(metrics)

        fig_cm, fig_imp = make_plots(m, thr)
        mlflow.log_figure(fig_cm, "plots/confusion_matrix.png")
        mlflow.log_figure(fig_imp, "plots/feature_importance.png")
        plt.close("all")
        mlflow.log_dict(meta, "meta/model_meta.json")

        sig_X = X_val.astype("float64")
        info = mlflow.xgboost.log_model(
            m, name="model",
            signature=infer_signature(sig_X, m.predict_proba(X_val)[:, 1]),
            input_example=sig_X.head(5))
    return {"variant": variant, "run_id": run.info.run_id, "model_uri": info.model_uri,
            "threshold": thr, "metrics": metrics}


results = {
    "baseline": log_variant("baseline", model, THRESHOLD,
                            len(y_train), float(y_train.mean())),
    "smote":    log_variant("smote", model_smote, THRESHOLD_SMOTE,
                            len(y_train_res), float(y_train_res.mean())),
}

summary = pd.DataFrame({k: v["metrics"] for k, v in results.items()}).T
summary[["val_f1", "val_pr_auc", "test_precision", "test_recall", "test_f1", "test_pr_auc"]].round(4)

In [ ]:
# Pilih champion lalu daftarkan ke Model Registry
if MODEL_TO_REGISTER == "auto":
    chosen = max(results, key=lambda k: results[k]["metrics"]["val_f1"])
else:
    chosen = MODEL_TO_REGISTER
best = results[chosen]
print(f"Champion: {chosen} (val_f1={best['metrics']['val_f1']:.3f}, "
      f"test_pr_auc={best['metrics']['test_pr_auc']:.3f}, threshold={best['threshold']:.3f})")

mv = mlflow.register_model(best["model_uri"], REGISTERED_MODEL_NAME)
tags = {
    "threshold": f"{best['threshold']:.6f}",
    "features": json.dumps(FEATURES),
    "type_map": json.dumps(TYPE_MAP),
    "variant": chosen,
    "run_id": best["run_id"],
    "git_sha": GIT_SHA,
    "val_f1": f"{best['metrics']['val_f1']:.4f}",
    "test_pr_auc": f"{best['metrics']['test_pr_auc']:.4f}",
}
for k, v in tags.items():
    client.set_model_version_tag(REGISTERED_MODEL_NAME, mv.version, k, v)
client.set_registered_model_alias(REGISTERED_MODEL_NAME, MODEL_ALIAS, mv.version)
print(f"Terdaftar: models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS} -> version {mv.version}")

## 9. Inference 1 data baru (dari MLflow Registry)

Model dimuat dari registry lewat alias, persis seperti yang dilakukan FastAPI di produksi: bukan dari file lokal. Threshold dan daftar fitur diambil dari tag model version.

Data di bawah dirancang agar **seharusnya gagal** menurut aturan dataset: tipe L dengan `tool wear x torque` = 215 x 55 = 11.825, di atas ambang 11.000 (overstrain). Suhu dan daya masih normal, jadi ini menguji apakah model menangkap aturan overstrain, bukan sekadar menebak.

In [ ]:
new_sample = {
    "type": "L",
    "air_temp_k": 300.5,
    "process_temp_k": 310.9,
    "rpm": 1500,
    "torque_nm": 55.0,
    "tool_wear_min": 215,
}

model_uri = f"models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS}"
loaded = mlflow.xgboost.load_model(model_uri)
mv_loaded = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, MODEL_ALIAS)
feats = json.loads(mv_loaded.tags["features"])
thr = float(mv_loaded.tags["threshold"])

new_df = pd.DataFrame([new_sample])
new_X = make_features(new_df)[feats]

proba = float(loaded.predict_proba(new_X)[0, 1])
pred = int(proba >= thr)

print("Model               :", model_uri, f"(version {mv_loaded.version})")
print(new_X.T)
print(f"\nProbabilitas failure: {proba:.3f}")
print(f"Threshold           : {thr:.3f}")
print("Prediksi            :", "FAILURE" if pred else "NORMAL")
print("Ekspektasi aturan   : FAILURE (overstrain, 11.825 > 11.000 untuk tipe L)")

## Catatan untuk laporan

- Dataset sintetis dan labelnya dibentuk dari aturan ambang, jadi skor tinggi tidak otomatis berarti performa sama di data pabrik nyata. Tulis sebagai keterbatasan.
- `RNF` (kegagalan acak 0,1%) tidak bisa diprediksi dari sensor, sehingga recall 100% memang tidak realistis.
- Untuk perbandingan, ganti bagian 4 dengan Random Forest dan Logistic Regression lalu bandingkan PR-AUC, recall, dan F1 pada test set yang sama.
- Fungsi `make_features` dipakai juga di `serving/app.py`. Kalau rumus fitur diubah di notebook, ubah juga di API (daftar fitur & urutannya sudah otomatis ikut model lewat tag MLflow, tapi rumusnya tidak).